# Coffee Standard J25 — SAFE policy root-cause audit

No-training audit of target support, sampler exposure, and raw/final confusion for D0DIRECT, SAFED0, and AF2LUMSAFE at lambda=0. Locked test remains closed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
BRANCH='codex/j25-safe-policy-root-cause'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
experiment_roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
def find_root(folder, relative):
    matches=[root/folder for root in experiment_roots if (root/folder/relative).is_file()]
    if not matches: raise FileNotFoundError(f'{folder}/{relative} tidak ditemukan di Drive')
    return matches[0]
D0_ROOT=find_root('coffee-standard-j25-af2-direct-v2',Path('val_reports/D0DIRECT_seed42_result.json'))
CONTROL_ROOT=find_root('coffee-standard-j25-safe-d0-v1',Path('val_reports/SAFED0_seed42_result.json'))
SAFE_ROOT=find_root('coffee-standard-j25-af2-luminance-safe-v1',Path('val_reports/AF2LUMSAFE_seed42_result.json'))
STRENGTH_ROOT=find_root('coffee-standard-j25-af2-luminance-strength-sweep-v1',Path('af2_luminance_strength_sweep.json'))
PROJECT=D0_ROOT.parents[1]
print('D0:',D0_ROOT); print('CONTROL:',CONTROL_ROOT); print('SAFE:',SAFE_ROOT); print('PROJECT:',PROJECT)

In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
OUT=PROJECT/'experiments/coffee-standard-j25-safe-policy-audit-v1'; OUT.mkdir(parents=True,exist_ok=True)
SUMMARY=OUT/'safe_policy_root_cause.json'
print('DATA:',contract['images'],'| OUT:',OUT)

In [ ]:
D0_CHECKPOINT=D0_ROOT/'D0DIRECT/D0DIRECT_seed42/weights/best.pt'
D0_RESULT=D0_ROOT/'val_reports/D0DIRECT_seed42_result.json'
CONTROL_CHECKPOINT=CONTROL_ROOT/'SAFED0/SAFED0_seed42/weights/best.pt'
CONTROL_RESULT=CONTROL_ROOT/'val_reports/SAFED0_seed42_result.json'
SAFE_CHECKPOINT=SAFE_ROOT/'AF2LUMSAFE/AF2LUMSAFE_seed42/weights/best.pt'
SAFE_RESULT=SAFE_ROOT/'val_reports/AF2LUMSAFE_seed42_result.json'
STRENGTH=STRENGTH_ROOT/'af2_luminance_strength_sweep.json'
SAMPLER=CONTROL_ROOT/'sampler_audit.json'
required=(D0_CHECKPOINT,D0_RESULT,CONTROL_CHECKPOINT,CONTROL_RESULT,SAFE_CHECKPOINT,SAFE_RESULT,STRENGTH,SAMPLER)
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Artefak kurang: {missing}')
LOG=OUT/'safe_policy_root_cause_run.log'
command=[sys.executable,'-u','-m','coffee_detector.analysis.coffee_standard_j25_safe_policy_audit','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--d0-checkpoint',str(D0_CHECKPOINT),'--d0-result',str(D0_RESULT),'--control-checkpoint',str(CONTROL_CHECKPOINT),'--control-result',str(CONTROL_RESULT),'--safe-checkpoint',str(SAFE_CHECKPOINT),'--safe-result',str(SAFE_RESULT),'--strength-summary',str(STRENGTH),'--sampler-audit',str(SAMPLER),'--output',str(SUMMARY),'--device','0','--authorize-diagnostic']
print('MENJALANKAN VALIDATION-ONLY SAFE-POLICY AUDIT | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
minutes=0
while process.poll() is None:
    time.sleep(120); minutes+=2; print(f'Masih berjalan: {minutes} menit',flush=True)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'Audit gagal: {process.returncode}')
result=json.loads(SUMMARY.read_text())
print('TRAIN SUPPORT:',result['target_training_support'])
print('ATTRIBUTION:',result['attribution'])
print('NEXT:',result['next_ablation_priority'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])

In [ ]:
import pandas as pd
rows=[]
for model,stages in result['models'].items():
    for stage,values in stages.items():
        rows.append({'model':model,'stage':stage,**{key:values[key] for key in ('targets','accessible','matched','correct_class','wrong_class','proposal_accessibility','matched_recall','localization_conditioned_class_accuracy')},'wrong_destinations':values['wrong_destinations']})
display(pd.DataFrame(rows))
print('CO-OCCURRENCE:',result['target_training_support']['cooccurring_target_image_counts'])
print('NEXT:',result['next_ablation_priority'])
print('SUMMARY:',SUMMARY)